# 动手练习：动态功能连接与多层脑网络

来自《生物医学网络科学》（Biomedical Network Science）。

每个任务先运行“准备”格，再在“你的代码”格中完成 TODO，最后运行“核对”格：打印 ✓ 表示通过，否则会给出提示。数据文件放在与本笔记本同级的 `data/` 文件夹中。

## 任务 1：实现滑动窗口相关

本练习前 5 个任务使用 numpy 生成的**模拟数据**，只用于演示方法，不代表真实 fMRI。模拟的“BOLD”信号做了一阶自回归平滑（$\phi=0.8$），以模仿真实 BOLD 低频、强自相关的特点。

setup 生成两组两通道信号，各 600 个时间点：

- `A`（平稳）：全程真实相关 0.3；
- `B`（切换）：前 300 点真实相关 0.7，后 300 点真实相关 0。

写函数 `sliding_corr(x, y, w, step=1)`，返回每个窗口（长度 `w`）内的皮尔逊相关；计算窗长 40 时的 `rA`、`rB`。

期望看到：`rB` 在中点附近明显下降；但 `rA` 虽然真实相关恒定，窗口相关也在 −0.4 到 0.7 之间大幅起伏。

> **提示：** 最直接的写法是逐窗调用 `np.corrcoef(x[t:t+w], y[t:t+w])[0, 1]`；窗口数为 $T-w+1$。

In [ ]:
# 准备（直接运行）
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.vq import kmeans2


def ar1_smooth(e, phi=0.8):
    out = np.zeros_like(e)
    for t in range(1, e.shape[-1]):
        out[..., t] = phi * out[..., t - 1] + e[..., t]
    return out


def correlated_pair(rho, T, rng):
    """两通道模拟信号，真实相关为 rho（模拟数据）。"""
    z = rng.standard_normal((2, T))
    return ar1_smooth(np.vstack([z[0], rho * z[0] + np.sqrt(1 - rho ** 2) * z[1]]))


rng = np.random.default_rng(1)
A = correlated_pair(0.3, 600, rng)
B = np.hstack([correlated_pair(0.7, 300, rng), correlated_pair(0.0, 300, rng)])

In [ ]:
def sliding_corr(x, y, w, step=1):
    # TODO
    return None


rA = None
rB = None

In [ ]:
# 核对（不用修改）
assert rA is not None and rB is not None, "rA、rB 还没有计算"
assert len(rA) == 561, "600 个点、窗长 40、步长 1 应得到 600 − 40 + 1 = 561 个窗口"
assert abs(rA[0] - np.corrcoef(A[0, :40], A[1, :40])[0, 1]) < 1e-9, "第一个窗口应是前 40 个点的相关"
assert rB[:200].mean() - rB[-200:].mean() > 0.4, "B 的前段窗口相关应明显高于后段"
assert rA.max() - rA.min() > 0.8, "A 的窗口相关应有大幅起伏（真实相关恒定，起伏来自有限样本）"
print("✓ 任务 1 通过")

## 任务 2：窗长的权衡

对平稳信号 `A` 分别用窗长 20、40、80、160 计算窗口相关的标准差，存入列表 `sd_by_w`（与 `windows` 顺序一致）。

期望看到：窗越短，纯采样波动越大。即使窗长 160，波动仍不可忽略，因为强自相关让**有效样本数**远小于时间点数。对照专题页的估算：独立样本时 $\operatorname{SE}(z)\approx1/\sqrt{w-3}$。

In [ ]:
windows = [20, 40, 80, 160]
sd_by_w = []   # TODO

In [ ]:
# 核对（不用修改）
assert len(sd_by_w) == 4, "sd_by_w 应有 4 个值"
assert all(a > b for a, b in zip(sd_by_w, sd_by_w[1:])), "窗长越长，窗口相关的标准差应越小"
assert abs(sd_by_w[1] - 0.239) < 0.01, "窗长 40 时标准差应约为 0.24"
assert sd_by_w[-1] > 1 / np.sqrt(160 - 3), "窗长 160 的实际波动应大于独立样本的估算——自相关减少了有效样本数"
print("✓ 任务 2 通过")

## 任务 3：相位随机化零模型

写函数 `phase_randomize(X, rng)`：对多通道信号 `X`（通道 × 时间）做实数 FFT，给**所有通道乘以同一组**随机相位因子 $e^{i\varphi_f}$（零频和奈奎斯特频率的相位设为 0），再逆变换。这样保留了各通道功率谱和通道间的静态相关，破坏随时间变化的耦合。

以窗长 40 的窗口相关标准差为统计量，对 `A`、`B` 各生成 200 个替代数据，得到单侧 $p$ 值 `pA`、`pB`（$p=(\#\{\text{null}\ge\text{obs}\}+1)/201$）。使用 setup 里给定的 `rng3`。

期望看到：`A` 不超出零分布（没有“动态”），`B` 明显超出。

> **提示：** `np.fft.rfft(X, axis=1)`、`np.fft.irfft(F, n=T, axis=1)`。

In [ ]:
# 准备（直接运行）
rng3 = np.random.default_rng(5)

In [ ]:
def phase_randomize(X, rng):
    # TODO
    return None


pA, pB = None, None

In [ ]:
# 核对（不用修改）
S_chk = phase_randomize(B, np.random.default_rng(0))
assert S_chk is not None and S_chk.shape == B.shape, "替代数据的形状应与原数据相同"
assert np.allclose(np.abs(np.fft.rfft(S_chk, axis=1)), np.abs(np.fft.rfft(B, axis=1))), "替代数据应保留每个通道的功率谱（傅里叶幅值不变）"
assert abs(np.corrcoef(S_chk)[0, 1] - np.corrcoef(B)[0, 1]) < 1e-6, "两个通道必须乘以同一组随机相位，才能保留静态相关"
assert pA is not None and pA > 0.05, "平稳信号 A 不应超出平稳零模型"
assert pB is not None and pB < 0.05, "切换信号 B 应显著超出平稳零模型"
print("✓ 任务 3 通过")

## 任务 4：状态序列的概括指标

写函数 `state_summary(seq, s)`，对状态序列 `seq` 中的状态 `s` 返回三元组 `(占时比例, 平均停留时间, 转移概率 P(s→s))`，再返回整条序列的切换次数可以另写一行。用专题页的算例核对：序列 `1 1 1 2 2 1 1 3 3 3` 中状态 1 的占时 0.5、平均停留 2.5 个窗口、$\hat P(1\to1)=3/5$，切换 3 次。

计算 `summary1 = state_summary(example, 1)` 和 `n_switch`。

> **提示：** 转移概率的分母是“处于状态 s 且有下一个窗口”的次数，不含最后一个窗口。

In [ ]:
# 准备（直接运行）
example = [1, 1, 1, 2, 2, 1, 1, 3, 3, 3]

In [ ]:
def state_summary(seq, s):
    # TODO
    return None


summary1 = None
n_switch = None

In [ ]:
# 核对（不用修改）
assert summary1 is not None, "summary1 还没有计算"
assert abs(summary1[0] - 0.5) < 1e-9, "状态 1 的占时比例应为 5/10 = 0.5"
assert abs(summary1[1] - 2.5) < 1e-9, "状态 1 出现两段（长 3 和 2），平均停留 2.5"
assert abs(summary1[2] - 0.6) < 1e-9, "P(1→1) = 3/5；分母不含最后一个窗口"
assert n_switch == 3, "整条序列切换 3 次（1→2、2→1、1→3）"
assert abs(state_summary([2, 2, 1, 1, 1, 2, 3, 3, 2, 2], 2)[1] - 5 / 3) < 1e-9, "再核对一例：状态 2 三段长 2、1、2，平均停留 5/3"
print("✓ 任务 4 通过")

## 任务 5：从窗口到“连接状态”，以及 k-means 的陷阱

setup 模拟 6 个脑区、900 个时间点，隐藏状态按 S1 → S2 → S1 → S2 交替（每段 250、200、250、200 点）：S1 时脑区 {0,1,2}、{3,4,5} 各自共变；S2 时 {0,3}、{1,4}、{2,5} 共变。

1. 用窗长 40、步长 5 计算每个窗口的 15 条边相关，组成矩阵 `V`（窗口 × 边）；
2. 用 `kmeans2(V, 2, seed=0, minit="++")` 聚类，与每个窗口的真实状态 `truth_w` 比较，得到一致率 `agree`（标签编号任意，取两种对应中较高的一种）；
3. 对整段数据做一次相位随机化（用 `rng5`），在替代数据上重复同样的流程，得到两个“状态”的占时 `occ_surrogate`。

期望看到：`agree` 很高，但交界处的窗口会混合两种模式；在没有任何状态切换的平稳替代数据上，k-means 仍然“找到”两个占时可观的状态——它总会返回你指定数目的聚类中心。

In [ ]:
# 准备（直接运行）
def segment(state, T, rng):
    f = rng.standard_normal((3, T))
    Xs = rng.standard_normal((6, T))
    groups = [[0, 1, 2], [3, 4, 5]] if state == 0 else [[0, 3], [1, 4], [2, 5]]
    for g, members in enumerate(groups):
        for m in members:
            Xs[m] += 1.2 * f[g]
    return ar1_smooth(Xs)


rng4 = np.random.default_rng(3)
lengths, states = [250, 200, 250, 200], [0, 1, 0, 1]
Xst = np.hstack([segment(s, L, rng4) for s, L in zip(states, lengths)])
truth_t = np.concatenate([[s] * L for s, L in zip(states, lengths)])
w5, step5 = 40, 5
iu6 = np.triu_indices(6, k=1)
starts5 = list(range(0, Xst.shape[1] - w5 + 1, step5))
truth_w = np.array([np.round(truth_t[t:t + w5].mean()) for t in starts5])
rng5 = np.random.default_rng(9)

In [ ]:
V = None
agree = None
occ_surrogate = None

In [ ]:
# 核对（不用修改）
assert V is not None and V.shape == (173, 15), "V 应为 173 个窗口 × 15 条边；检查窗口起点 range(0, 900 − 40 + 1, 5)"
assert agree is not None and agree > 0.9, "k = 2 应能恢复大部分窗口的真实状态（一致率 > 0.9）"
assert occ_surrogate is not None and abs(sum(occ_surrogate) - 1) < 1e-9, "occ_surrogate 应是两个状态的占时比例，和为 1"
assert min(occ_surrogate) > 0.2, "平稳替代数据上 k-means 仍会给出两个都有可观占时的“状态”"
print("✓ 任务 5 通过")

## 任务 6：小分析：真实 EEG 的滑动窗口 α 同步

用真实数据 `data/eeg_eyes_open_closed.csv`（PhysioNet EEG Motor Movement/Imagery Dataset，ODC-By 1.0；6 名被试睁眼 EO、闭眼 EC 各 60 秒，19 个电极，80 Hz；出处见本讲真实数据案例与 `data/SOURCES.md`）。setup 已经读入数据，并给出带通滤波和希尔伯特相位的函数 `alpha_phase(X)`。

选一名被试 `subject`（如 `"S001"`）和一个窗长 `win_sec`（1–5 秒），对 EO 和 EC 两段分别计算每个窗口（步长 1 秒）内 171 对电极的相位锁定值 PLV，再对所有电极对取平均，得到全网平均 PLV 的时间序列 `plv_eo`、`plv_ec`。

画出两条曲线，并思考：闭眼时同步更强吗？换一个 α 很弱的被试（S005、S006）呢？窗长变短时曲线怎样变化？这些起伏有多少是真实的“动态”——你会用什么零模型来检验？

In [ ]:
# 准备（直接运行）
import pandas as pd
from scipy.signal import butter, filtfilt, hilbert

eeg = pd.read_csv("data/eeg_eyes_open_closed.csv")
channels = list(eeg.columns[2:])
fs = 80
_b, _a = butter(4, [8 / (fs / 2), 12 / (fs / 2)], btype="band")


def get_segment(subject, condition):
    g = eeg[(eeg.subject == subject) & (eeg.condition == condition)]
    Xg = g[channels].to_numpy(float).T
    return Xg - Xg.mean(0)                    # 共同平均参考


def alpha_phase(Xg):
    """8–12 Hz 零相位带通 + 希尔伯特相位，去掉两端各 1 秒。"""
    return np.angle(hilbert(filtfilt(_b, _a, Xg, axis=1), axis=1))[:, fs:-fs]

In [ ]:
subject = "S001"
win_sec = 2
plv_eo = None
plv_ec = None

In [ ]:
# 核对（不用修改）
assert subject in set(eeg.subject), "subject 应是 S001–S006 之一"
assert 1 <= win_sec <= 5, "win_sec 请取 1–5 秒"
assert plv_eo is not None and plv_ec is not None, "plv_eo、plv_ec 还没有计算"
assert len(plv_eo) == len(plv_ec) and len(plv_eo) >= 50, "两段的窗口数应相同（58 秒有效数据、步长 1 秒，约 54–57 个窗口）"
assert np.all((np.asarray(plv_eo) >= 0) & (np.asarray(plv_eo) <= 1)) and np.all((np.asarray(plv_ec) >= 0) & (np.asarray(plv_ec) <= 1)), "PLV 应在 0 到 1 之间"
print("✓ 任务 6 通过")